# Registry

To update the `registry` *(registry.csv)*:

```python
operator.ingest_data(mode='slide', data_classes=['samples'])
operator.ingest_data(mode='tile', data_classes=['samples'])
```

- this creates a unique uuid for each `slide_path`, 
- identical `slide_names` is fine as long as their **paths are different** in the database

Example `registry.csv`:
```csv
slide_id,slide_name,slide_path,slide_class
f6164cbba4f94a7c86fa5bb80bb0b986,TCGA-4N-A93T-01Z-00-DX2.svs,colon/TCGA-4N-A93T-01Z-00-DX2.svs,colon
83c4a068dac0408290c2a7e248fc9ab1,TCGA-5M-AAT6-01Z-00-DX1.svs,colon/TCGA-5M-AAT6-01Z-00-DX1.svs,colon
29b8038f0c0d45d28ea8f8a31b7dbb9e,TCGA-G9-6353-01Z-00-DX1.svs,prostate/TCGA-G9-6353-01Z-00-DX1.svs,prostate
e908537e975b407799b6c345574b8be2,002.tiff,samples/002.tiff,samples
e8e3e80127da46ed81bb184ca95456ad,001.tiff,samples/001.tiff,samples
```

## Slide Database

```

SLIDE_DATABASE/
├── ARTIFACTS/
│   ├── slide_{uuid1}/
│   │   ├── {slide_name}.json
│   │   ├── {slide_name}_extractions.h5
│   │   ├── {slide_name}_slide_thumbnail.png
│   │   ├── {slide_name}_masked_thumbnail.png
│   │   ├── {slide_name}_normalised_thumbnail.png
│   ├── slide_{uuid2}/
│   │   └── ...
│   └── ...
│
├── DATASETS/
│   ├── colon/
│   │   ├── TCGA_01.svs
│   │   └── ...
│   ├── prostate/
│   │   └── ...
│   ├── samples/
│   │   ├── 001.tiff
│   │   └── ...
│   └── ...
|
├── registry.csv

```


The **manifest** file for each **artifact**: 

`{slide_name}.json`
1. extractions will be created for different `ext_configs` and `coords_configs`
2. except for the cases that `ext_patch_strategy` is in `keep` or `aggr`
3. there is no reason to generate identical extractions when its just to `aggr` *(aggregate)* or `keep` patch tokens
4. by default, `ext_patch_strategy` is `discard` for tile-only extractions and save disk space

Heres an example:
```json
{
  "slide_thumbnail": {
    "overview_mpp": 8.0
  },
  "masked_thumbnail": {
    "overview_mpp": 8.0,
    "method": "morphological"
  },
  "extractions": {
    "ext_1": {
      "ext_configs": {
        "ext_enc": 224,
        "ext_mpp": 0.5,
        "ext_model": "hop0",
        "ext_patch_strategy": "aggr"
      },
      "coords_configs": {
        "method": "morphological",
        "ann_mask": false,
        "tile_threshold": 0.25,
        "stride": 0.0
      },
      "feats_configs": {
        "window_level": "tile",
        "patch_to_tile": "discard",
        "grid_size": 2
      }
    }
  },
  "extractions": {
    "ext_2": {
      "ext_configs": {
        "ext_enc": 224,
        "ext_mpp": 0.5,
        "ext_model": "hop0",
        "ext_patch_strategy": "discard"
      },
      "coords_configs": {
        "method": "morphological",
        "ann_mask": true,
        "tile_threshold": 0.25,
        "stride": 0.0
      },
      "feats_configs": {
        "window_level": "patch",
        "patch_to_tile": "keep",
        "grid_size": 2
      }
    }
  }
}
```

## Tile Database

```

TILE_DATABASE/
├── ARTIFACTS/
│   ├── lung_colon/
│   │   ├── lung_colon.json
│   │   ├── lung_colon_extractions.h5
│   │   ├── lung_colon_ext_1.csv
│   │   ├── lung_colon_ext_2.csv
│   ├── pancancer/
│   │   └── ...
│   └── ...
│
├── DATASETS/
│   ├── lung_colon/
│   │   ├── colon_ade_colonca2.jpeg
│   │   └── ...
│   ├── pancancer/
│   │   └── ...
│   ├── sicap/
│   │   ├── Block_Region_1.jpg
│   │   └── ...
│   └── ...
|
├── registry.csv

```


The **manifest** file for each **artifact**: 

Note: `tile_class` is the **original folder the tile dataset reside**, using it as a reference for **artifact** too

`{tile_class}.json`
1. extractions will be created for different `ext_configs` and `coords_configs`
2. even for the cases that `ext_patch_strategy` is in `keep` or `aggr`
3. we dont want to corrupt chunks by updating **different patch strategies** in a chunk
4. `ext_type` have options of `standalone` and `windowed`
5. this allows tiles to be processed with the **window mechanic** *(similar to slides)*
6. thus, we require `base mpp` for windowing tiles

Heres an example:
```json
{
  "base_mpp": 0.5,
  "extractions": {
    "ext_1": {
      "ext_configs": {
        "ext_enc": 224,
        "ext_mpp": 0.5,
        "ext_model": "hop0",
        "ext_patch_strategy": "aggr",
        "ext_type": "standalone"
      }
    },
    "ext_2": {
      "ext_configs": {
        "ext_enc": 224,
        "ext_mpp": 0.5,
        "ext_model": "hop0",
        "ext_patch_strategy": "aggr",
        "ext_type": "windowed"
      }
    }
  }
}
```


`{tile_class}_{ext_id}.csv`

1. for each **new extractions**, a new csv is created for clarity
2. we split `embeddings` into chunks of `chunk_size` (generally 10,000)
3. then note down their `lcoal_id` and `chunk_id` here


```csv
tile_id,chunk_id,local_id,tile_path
7b53012516464a8d91bed62f7248a2f6,0,0,/mnt/a/APEIRON_v1/DATABASE/TILE_DATABASE/DATASETS/lung_colon/colon_ade_colonca93.jpeg
2f0394b2c11146c390960f746ba32fb4,0,1,/mnt/a/APEIRON_v1/DATABASE/TILE_DATABASE/DATASETS/lung_colon/colon_ade_colonca95.jpeg
9614cd67200e404189b8b8a6d6b0a55a,0,2,/mnt/a/APEIRON_v1/DATABASE/TILE_DATABASE/DATASETS/lung_colon/colon_ade_colonca98.jpeg
```

## Client directories

**Project Root**

```

project_root/
├── user1/
│   ├── task1/
│   │   ├── normalise_targets/
│   │   │   ├── macenko_sample.png
│   │   │   ├── sample_1.png
│   │   │   ├── sample_2.png
│   │   │   ├── sample_3.png
│   │   │   └── ...
│   │   ├── annotations/
│   │   │   ├── subset1.tif
│   │   │   ├── subset2.tif
│   │   │   └── ...
│   │   ├── config.yaml
│   │   ├── slide_dataset.csv
│   │   └── tile_dataset.csv
│   ├── task2/
│   │   ├── config.yaml
│   │   ├── slide_dataset.csv
│   │   ├── tile_dataset.csv
│   │   └── ...
│
├── user2/
│   ├── task1/
│   │   ├── config.yaml
│   │   └── slide_dataset.csv
│   │   └── tile_dataset.csv
│   └── ...
└── ...

```

**Models Root**

```

models_root/
├── foundational_models/
│   ├── hop0/
│   │   ├── hop0.pth
│   │   └── hop0_transform.pkl
│   ├── vir1/
│   │   ├── vir1.pth
│   │   └── vir1_transform.pkl
│   └── ...

```

put `config.yaml` into the **project folder**

```yaml
overview_mpp: 8.0 # thumbnails (visuals) generates at 'micron per pixel' 

norm_configs: # the normalise method and target (in the folder: normalise_targets) for targeted regions analysis
  'method': 'macenko'
  'target_name': 'sample2.png'
  
slide:
  ext_configs:
    'ext_enc': 224  # the backbone model extracts at 'encoder', window size
    'ext_mpp': 0.5  # the backbone model extracts at 'micron per pixel'
    'ext_model': "hop0"  # ['hop0', 'hop1', 'vir1', 'vir2', 'ch15', 'uni2h', 'mstar', 'dino3']
    'ext_patch_strategy': 'discard'  # ['aggr', 'keep', 'discard']
  coords_configs:
    'method': 'morphological'  # the type of masking applied
    'ann_mask': True # Use annotations as masks
    'tile_threshold': 0.25  # ratio of tissue to determine as a viable tile
    'stride': 0.0  # 0.0~1.0 to determine how much of neighbour tiles are combined
  feats_configs:
    'window_level': 'tile'  # ['grid', 'tile', 'patch']
    'patch_to_tile': 'discard'  # ["max", "mean", "discard"]
    'grid_size': 2  # how many tiles to to create a grid (e.g. 2x2 tiles)

tile:
  ext_configs:
    'ext_enc': 224
    'ext_mpp': 0.5
    'ext_model': "hop0"
    'ext_patch_strategy': 'keep'
    'ext_type': 'standalone'  # either as standalone or windowed, windowed turns each tile into pseudo-slide
  coords_configs:
    'stride': 0.0
  feats_configs:
    'window_level': 'tile'  # only patch or tile, no grid support (redundant)
    'patch_to_tile': 'mean'
    'grid_size': 2

ground_truth: # GT labels for training
  ann_configs: # annotations provided in project folder
    'ann_type': 'shape' # shape or pixel, shape is json annotation of polygons and ellipses while pixel is binary or multi-class binary mask
    'crop_ann': False   # Bag regions of Interest from the json label (only available for shape)
    'class_id_map':
      '0': 'background'
      '1': 'gleason_3'
      '2': 'gleason_4'
      '3': 'gleason_5'
      '4': 'normal'
      '5': 'stroma'
  label_configs: # labels provided in the slide_dataset and tile_dataset csvs (multi class binary)
    'label_id_map':
      '0': 'background'
      '1': 'gleason_3'
      '2': 'gleason_4'
      '3': 'gleason_5'
      '4': 'normal'
      '5': 'stroma'
```

put `dataset.csv` into the **project folder**

1. **class columns** are **appendable** *(class_0, class1)*, each are of [0.0~1.0]
2. flexible to *binary classifications* and *scalar values*

`slide mode:`
```csv
slide_id,slide_name,class0, class1
e8e3e80127da46ed81bb184ca95456ad,001.tiff,1,0.3
e908537e975b407799b6c345574b8be2,002.tiff,0,0.5
```

`tile mode:`
```csv
tile_id,tile_name,class0
7b53012516464a8d91bed62f7248a2f6,colon_ade_colonca93.jpeg,
2f0394b2c11146c390960f746ba32fb4,colon_ade_colonca95.jpeg,
9614cd67200e404189b8b8a6d6b0a55a,colon_ade_colonca98.jpeg,
cf6bbd13c03e4502b4fc779791e5b966,colon_non-tumour_TCGA-3L-AA1B-01Z-00-DX1_10752_27136.jpg,
be2fc8c82ed94c53887dd6301a6a49dc,colon_non-tumour_TCGA-3L-AA1B-01Z-00-DX1_11264_25600.jpg,
```
